<a href="https://colab.research.google.com/github/Zeenith24/Gen-AI-College/blob/main/GenAIExp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython scikit-learn transformers sentencepiece


  Using cached biopython-1.88-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.4 MB/s eta 0:00:00


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from Bio import Entrez
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Entrez.email = "ydskjfskjfksdjf@gmail.com"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok = AutoTokenizer.from_pretrained('google/flan-t5-small')
# Revert tie_word_embeddings to restore proper model weights
mod = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-small').to(device)

def biomedical_qa(question, search_term=None, n_docs=10, top_k=3):
    search_term = search_term or question
    # 1. Fetch & parse
    ids = Entrez.read(Entrez.esearch(db="pubmed", term=search_term, retmax=n_docs))['IdList']
    if not ids: return "No articles found."
    raw_text = Entrez.efetch(db="pubmed", id=",".join(ids), rettype="abstract", retmode="text").read()
    arts = [a.strip() for a in raw_text.split('\n\n') if len(a.strip()) > 100][:n_docs]

    # 2. Rank (TF-IDF)
    vect = TfidfVectorizer(stop_words="english")
    doc_vects = vect.fit_transform(arts)
    sims = cosine_similarity(vect.transform([question]), doc_vects)[0]
    context = " ".join([arts[i] for i in sims.argsort()[-top_k:][::-1]])

    # 3. QA
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inps = tok(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    return tok.decode(mod.generate(**inps, max_length=100)[0], skip_special_tokens=True)

# Run
query = "What is the role of ACE2 in COVID-19?"
search_keywords = "ACE2 COVID-19"  # Better for PubMed search
print(f"Q: {query}\nA: {biomedical_qa(question=query, search_term=search_keywords)}")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Q: What is the role of ACE2 in COVID-19?
A: ACE2-like catalytic activity may be attributed to antibodies.
